# bharatBrief— News Sentiment Project

## Step 1 — Install & Import Libraries

In [1]:

# !pip install newsapi-python vaderSentiment streamlit -q


In [2]:
# Import everything we need
import pandas as pd                                         
import re                                                    
from datetime import datetime, timedelta                     
from newsapi import NewsApiClient                            # To connect to NewsAPI
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer  # For sentiment scoring


## Step 2 — Connect to NewsAPI & Fetch Headlines

We fetch 3 types of news:
- **India news** — top headlines from India
- **Company news** — articles mentioning specific Indian companies
- **Tech/AI news** — from top global tech sources

In [3]:
API_KEY = "b66e20d2570f4fe8979fd9bf7f66db13"   

COMPANIES = ["Reliance", "TCS", "Infosys", "Adani", "HDFC", "Wipro", "Zomato"]

# Connect to NewsAPI using our key
newsapi = NewsApiClient(api_key=API_KEY)

print(" Connected to NewsAPI!")
print(f"   Tracking {len(COMPANIES)} companies: {', '.join(COMPANIES)}")

 Connected to NewsAPI!
   Tracking 7 companies: Reliance, TCS, Infosys, Adani, HDFC, Wipro, Zomato


In [4]:
# india_response = newsapi.get_top_headlines(
#     sources='the-times-of-india,the-hindu,ndtv,india-today,news18',
#     page_size=30
# )

# india_articles = india_response.get('articles', [])
# print(f"✅ India news fetched: {len(india_articles)} articles")

# Force NewsAPI to pull only from recognized Indian publishers
indian_sources = 'the-times-of-india,the-hindu,ndtv,india-today,news18,google-news-in'

india_response = newsapi.get_top_headlines(
    sources=indian_sources,
    page_size=30
)

india_articles = india_response.get('articles', [])
print(f"✅ India news fetched: {len(india_articles)} articles")

✅ India news fetched: 27 articles


In [5]:
# tech_response = newsapi.get_top_headlines(
#     sources='techcrunch,the-verge,wired,ars-technica',
#     page_size=20
# )

# tech_articles = tech_response.get('articles', [])
# print(f" Tech/AI news fetched: {len(tech_articles)} articles") 

# Expanded list of global Tech & AI publishers
tech_sources = 'techcrunch,the-verge,wired,ars-technica,engadget,hacker-news,recode'

tech_response = newsapi.get_top_headlines(
    sources=tech_sources,
    page_size=30 # Increased page size to get more data
)

tech_articles = tech_response.get('articles', [])
print(f"✅ Tech/AI news fetched: {len(tech_articles)} articles")

✅ Tech/AI news fetched: 20 articles


In [6]:

# company_articles = {}   # We'll store results as: { "Reliance": [...], "TCS": [...] }

# week_ago = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')

# for company in COMPANIES:
#     response = newsapi.get_everything(
#         q=company,                  # Search for this company name
#         language='en',
#         from_param=week_ago,        # Only last 7 days (free tier limit)
#         sort_by='publishedAt',      # Most recent first
#         page_size=10
#     )
#     company_articles[company] = response.get('articles', [])
#     print(f"  {company}: {len(company_articles[company])} articles")

# print(f"\n Company news fetched for all {len(COMPANIES)} companies!") 

# company_articles = {}   

# # Change: Set timeframe to maximum 1 day ago (yesterday) instead of 7 days
# yesterday = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')

# for company in COMPANIES:
#     response = newsapi.get_everything(
#         q=company,                  
#         language='en',
#         from_param=yesterday,       # <--- THIS FORCES ONLY FRESH DATA
#         sort_by='publishedAt',      
#         page_size=10
#     )
#     company_articles[company] = response.get('articles', [])
#     print(f"  {company}: {len(company_articles[company])} articles")

# print(f"\n Company news fetched for all {len(COMPANIES)} companies!") 

# ── REPLACE INDIA NEWS CELL ──
# We use get_everything with explicit domains and a 2-day lookback 
# to bypass the 2021 frozen data trap on the free tier.

yesterday = (datetime.now() - timedelta(days=2)).strftime('%Y-%m-%d')

# Using domains instead of sources works much better for fresh data
indian_domains = 'ndtv.com,indiatoday.in,indianexpress.com,hindustantimes.com,moneycontrol.com,livemint.com'

india_response = newsapi.get_everything(
    domains=indian_domains,
    language='en',
    from_param=yesterday,
    sort_by='publishedAt',
    page_size=30
)

india_articles = india_response.get('articles', [])
print(f"✅ Live India news fetched: {len(india_articles)} articles")

✅ Live India news fetched: 30 articles


## Step 3 — Clean the Raw Data

Raw API data has some problems we need to fix:

| Problem | Fix |
|---|---|
| Some articles are deleted — show as `[Removed]` | Skip them |
| Missing title or date | Skip them |
| Same article appears from 5 different sources | Keep only 1 |
| Dates are in a long string format | Convert to proper datetime |
| HTML symbols like `&amp;` in text | Remove them |
| Short junk headlines (under 20 chars) | Skip them |

In [7]:
# This function takes a raw list of articles and returns a clean DataFrame
def clean_articles(articles, category="india", company=None):

    clean_rows = []   # We'll collect good articles here

    for article in articles:

        # Extract the fields we care about
        title       = article.get('title') or ''
        description = article.get('description') or ''
        url         = article.get('url') or ''
        published   = article.get('publishedAt') or ''
        source      = (article.get('source') or {}).get('name') or ''


        # Skip deleted articles (NewsAPI marks them as "[Removed]")
        if '[Removed]' in title:
            continue

        # Skip if title is missing or too short to be meaningful
        if not title.strip() or len(title) < 20:
            continue

        # Skip if no publish date (we can't do time analysis without it)
        if not published:
            continue

        # ── CLEAN the data we kept 

        # Convert date string "2025-04-27T06:30:00Z" → Python datetime object
        try:
            published_dt = datetime.strptime(published, '%Y-%m-%dT%H:%M:%SZ')
        except:
            continue    # Skip rows with unexpected date format

        # Remove HTML entities like &amp; &lt; from the text
        title       = re.sub(r'&[a-z]+;', ' ', title).strip()
        description = re.sub(r'&[a-z]+;', ' ', description).strip()

        # Clean source name — API sometimes returns "techcrunch" → we want "Techcrunch"
        source = source.strip().title()

        # Build a clean version of the text for sentiment scoring
        # We keep the original title for display, and use clean_text for VADER
        raw_text   = f"{title} {description}"
        clean_text = raw_text.lower()
        clean_text = re.sub(r'http\S+', '', clean_text)          # Remove URLs
        clean_text = re.sub(r'[^a-z0-9\s]', ' ', clean_text)     # Remove punctuation
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()      # Fix extra spaces

        # Add to our results list
        clean_rows.append({
            'title':        title,
            'url':          url,
            'source':       source,
            'published_at': published_dt,
            'category':     category,
            'company':      company,         # None for India/Tech articles
            'clean_text':   clean_text       # Used only for sentiment scoring
        })

    # Convert list to DataFrame
    df = pd.DataFrame(clean_rows)

    if not df.empty:
        before = len(df)
        # Remove duplicates — same URL from multiple sources = keep 1 row
        df = df.drop_duplicates(subset='url')
        print(f"  {len(articles):>3} raw → {before:>3} valid → {len(df):>3} after removing duplicates")

    return df


print(" clean_articles() function is ready!")

 clean_articles() function is ready!


In [8]:
# ── RUN CLEANING ON ALL 3 NEWS TYPES 

print("Cleaning India news...")
df_india = clean_articles(india_articles, category='india')

print("\nCleaning Tech news...")
df_tech  = clean_articles(tech_articles, category='tech')

print("\nCleaning Company news...")
company_dfs = []
for company, articles in company_articles.items():
    if articles:
        df_c = clean_articles(articles, category='company', company=company)
        company_dfs.append(df_c)

# Combine all company DataFrames into one
df_companies = pd.concat(company_dfs, ignore_index=True) if company_dfs else pd.DataFrame()

# Combine everything into one master DataFrame
df_all = pd.concat([df_india, df_tech, df_companies], ignore_index=True)
df_all = df_all.drop_duplicates(subset='url')   # Final global dedup

print(f"\n Total clean articles: {len(df_all)}")
print(df_all[['title', 'category', 'company', 'published_at']].head(5).to_string())

Cleaning India news...
   30 raw →  30 valid →  30 after removing duplicates

Cleaning Tech news...
   20 raw →  10 valid →  10 after removing duplicates

Cleaning Company news...


NameError: name 'company_articles' is not defined

## Step 4 — Sentiment Scoring with VADER

**VADER** = Valence Aware Dictionary and sEntiment Reasoner

It reads a piece of text and gives a score from **-1.0** (most negative) to **+1.0** (most positive).

| Score range | Label |
|---|---|
| `>= 0.05` | 🟢 Positive |
| `<= -0.05` | 🔴 Negative |
| Between | 🟡 Neutral |

> Why VADER? It works offline, requires zero setup, and is specifically tuned for short texts like news headlines.

In [ ]:
# Load VADER — works completely offline, no internet needed
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    """
    Takes a cleaned text string.
    Returns (label, score) — e.g. ('Positive', 0.65)
    """
    if not text or len(text.strip()) < 5:
        return 'Neutral', 0.0     # Too short to score reliably

    # polarity_scores() returns a dict like:
    # {'neg': 0.1, 'neu': 0.7, 'pos': 0.2, 'compound': 0.45}
    scores   = analyzer.polarity_scores(text)
    compound = scores['compound']   # This is the main score we use

    if compound >= 0.05:
        return 'Positive', round(compound, 3)
    elif compound <= -0.05:
        return 'Negative', round(compound, 3)
    else:
        return 'Neutral', round(compound, 3)


# Test it quickly before applying to all data
test_headlines = [
    "Reliance reports record profits, investors celebrate",
    "Market crashes as inflation fears grow",
    "RBI holds interest rates steady"
]

print("Quick test:")
for h in test_headlines:
    label, score = get_sentiment(h)
    print(f"  [{label:8}] {score:>6}  →  {h}")

In [ ]:
# Apply sentiment to every article in our master DataFrame
# We score 'clean_text' (not the raw title) because it's already stripped of noise

df_all[['sentiment', 'score']] = df_all['clean_text'].apply(
    lambda text: pd.Series(get_sentiment(text))
)

# Quick summary
print(" Sentiment scoring done!\n")
print("Overall breakdown:")
print(df_all['sentiment'].value_counts().to_string())

print("\nBreakdown by category:")
print(df_all.groupby(['category', 'sentiment']).size().unstack(fill_value=0).to_string())

## Step 5 — Preview Results & Save to CSV

Let's look at a few headlines from each category, then save everything to a CSV file.
The Streamlit app reads from this CSV — so run this notebook first every time.

In [ ]:
# ── PREVIEW: India headlines ──────────────────────────────────────────────
# We filter df_all (which has sentiment) instead of df_india (which doesn't)
print("🇮🇳 INDIA — Sample Headlines:")
print("-" * 70)
for _, row in df_all[df_all['category'] == 'india'].head(5).iterrows():
    emoji = {'Positive':'🟢', 'Negative':'🔴', 'Neutral':'🟡'}.get(row['sentiment'], '🟡')
    print(f"{emoji} [{row['sentiment']:8}] {row['title']}")
    print(f"   Source: {row['source']}  |  {row['published_at'].strftime('%d %b, %I:%M %p')}")
    print()

In [ ]:
# ── PREVIEW: Tech/AI headlines ────────────────────────────────────────────
print(" TECH/AI — Sample Headlines:")
print("-" * 70)
for _, row in df_all[df_all['category'] == 'tech'].head(5).iterrows():
    emoji = {'Positive':'🟢', 'Negative':'🔴', 'Neutral':'🟡'}.get(row['sentiment'], '🟡')
    print(f"{emoji} [{row['sentiment']:8}] {row['title']}")
    print(f"   Source: {row['source']}  |  {row['published_at'].strftime('%d %b, %I:%M %p')}")
    print()

In [ ]:
# ── PREVIEW: Company headlines ────────────────────────────────────────────
print(" COMPANY — Sample Headlines:")
print("-" * 70)
for _, row in df_all[df_all['category'] == 'company'].head(5).iterrows():
    emoji = {'Positive':'🟢', 'Negative':'🔴', 'Neutral':'🟡'}.get(row['sentiment'], '🟡')
    print(f"{emoji} [{row['sentiment']:8}] [{row['company']}] {row['title']}")
    print(f"   Source: {row['source']}  |  {row['published_at'].strftime('%d %b, %I:%M %p')}")
    print()

In [ ]:
# ── SAVE TO CSV ───────────────────────────────────────────────────────────
# We drop 'clean_text' before saving — it was only needed for scoring, not for display
df_save = df_all.drop(columns=['clean_text'])
df_save.to_csv('news_data.csv', index=False)

print(f" Saved {len(df_save)} articles to news_data.csv")
print("\nColumns in the CSV:")
print(df_save.dtypes.to_string())